# LES Smagorinsky — incompressible flow past a sphere

This notebook runs the finite-volume Navier–Stokes solver in `les_solver.py` on the
pre-processed unstructured tetrahedral mesh (`processed_mesh.npz`) and visualises the
flow past the sphere.

**Method**
- Cell-centred (collocated) unstructured finite volume.
- **SIMPLE** algorithm with implicit, under-relaxed momentum.
- **Rhie–Chow** face-flux interpolation for pressure–velocity coupling (avoids checkerboarding).
- **Smagorinsky** sub-grid-scale (LES) model for the effective viscosity `nu_eff = nu + nu_t`,
  with `nu_t = (Cs·Δ)²|S|`, `Δ = V^(1/3)`.
- Higher-order (deferred-correction) convection to resolve the recirculating wake.

**Boundary conditions**: inlet `u=(U,0,0)`; outlet `p=0`, `du/dn=0`; outer walls free-slip;
sphere no-slip.

Pipeline: `computational_grid_gmsh_visualized.ipynb` (mesh) →
`read_mesh_from_the_generated_mesh.ipynb` (`processed_mesh.npz`) → **this notebook**.

In [ ]:
import numpy as np
import les_solver

# Re = U * D / nu, with sphere diameter D = 1 m and U = 1 m/s -> Re = 100
u, p, mesh, g, hist = les_solver.run(
    nu=0.01,          # molecular kinematic viscosity  (Re = 100)
    U=1.0,            # inlet velocity
    iters=300,        # SIMPLE outer iterations
    Cs=0.17,          # Smagorinsky constant
    beta=0.9,         # high-order convection blend (0=upwind, 1=full high-order)
    limiter="none",   # 'none' (central, sharpest wake) | 'vanleer' | 'superbee'
    avg_last=150,     # report the field averaged over the last 150 iterations
    alpha_u=0.4, alpha_p=0.25,
    out_path="les_result.npz",
    log_every=50,
)
# `u`, `p` are the iteration-averaged (mean) fields, as is standard for LES.
# Instantaneous fields are also stored in les_result.npz (velocity_inst, pressure_inst).

In [ ]:
# Quick physical validation of the converged field
umag = np.linalg.norm(u, axis=1)
print(f"max |u|            = {umag.max():.3f} m/s   (flow acceleration around sphere)")
print(f"min |u|            = {umag.min():.3f} m/s   (~0 at the no-slip sphere surface)")
print(f"reversed-flow cells= {int(np.sum(u[:,0] < -0.02))}   (recirculating wake behind sphere)")
print(f"pressure range     = [{p.min():.3f}, {p.max():.3f}] Pa (low wake / high stagnation)")

In [ ]:
%matplotlib inline
from plot_les_result import plot_result

# Mid-plane (z = 2.5) velocity-magnitude + streamlines and pressure fields
plot_result("les_result.npz", "les_flow_past_sphere.png")

from IPython.display import Image
Image("les_flow_past_sphere.png")